## 1 Download Files

In [1]:
# download the data set
import kagglehub
import shutil

# Download dataset (default location)
path = kagglehub.dataset_download("khulasasndh/game-of-thrones-books")

# Move dataset to /content/
shutil.move(path, "/content/game-of-thrones-books")

print("Dataset moved to:", "/content/game-of-thrones-books")


100%|██████████| 3.71M/3.71M [00:01<00:00, 2.96MB/s]

Extracting files...


Error: Destination path '/content/game-of-thrones-books/1' already exists

In [2]:
import random

# Load your text data (assuming it’s a large string)
with open("/content/game-of-thrones-books/001ssb.txt", "r", encoding="utf-8") as file:
    text = file.readlines()  # Read file line by line (assuming each line is a sentence)

# Define the percentage of data to remove
drop_percentage = random.uniform(0.8, 0.9)  # Randomly select between 50% and 60%
num_to_remove = int(len(text) * drop_percentage)

# Randomly remove the lines
text = random.sample(text, len(text) - num_to_remove)  # Keep only the remaining sentences

In [3]:
type(text)

list

## 2 Text preprocessing

In [4]:
import string, re

In [5]:
def clean(text):
  sample = text
  sample = re.sub('[%s]' % re.escape(string.punctuation), '', sample)
  sample = [word for word in sample.split() if word.isalpha()]
  sample = [word.lower() for word in sample]
  sample = " ".join(sample)

  return sample

In [6]:
text = clean(str(text))

In [7]:
text

'use her as well weasels will tear out her entrails and carrion crows feast upon her eyes the flies off the n taken a great interest in the breeding of hunting hounds the lord had visited a master armorer to n the time he came trotting back id gotten a name out of her and a story she was a crofters child n when she took it out he died n it means i shall think on what you have said the maester told him firmly and now i believe i am n pyp was staring at jon as he got slowly to his feet his ears were red grenn grinning broadly did not n the watchman my lord jory said he vows hell never touch another horse n gave a shrug they never are n dark but i had a torch it made me so mad i almost gave him a swat in the head like old nan is always n suddenly doreah was tugging at her elbow my lady the handmaid whispered urgently your brother n there jon said he swung his horse around and galloped back across the bridge they watched him n held her ears again oh gods close the window n hold them ko pon

In [8]:
len(text) # total number of char in corpus

188727

In [9]:
# number of words in cleaned data
words = [word for word in text.split()]
print("Total number of words:",len(words))

Total number of words: 38062


In [10]:
# number of unique words
unique_word = len(set(words))
unique_word

4848

## 3 Make tocken

In [11]:
import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [12]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

In [13]:
vocab_len = len(tokenizer.word_index) # words with it's index
vocab_len

4848

In [14]:
# tokenizer.word_index

In [15]:
input_sequences = []
# 1 2 3 4

# inp opt
# 1    2
# 1 2  3 like wise

for sentence in text.split('\n'):
  tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0] # a b c --> [[0 1 2]] convert kari de

  for i in range(1, len(tokenized_sentence)):
    input_sequences.append(tokenized_sentence[:i+1])  # 0 to i

In [16]:
input_sequences[0:10]

[[562, 10],
 [562, 10, 17],
 [562, 10, 17, 82],
 [562, 10, 17, 82, 2362],
 [562, 10, 17, 82, 2362, 47],
 [562, 10, 17, 82, 2362, 47, 1576],
 [562, 10, 17, 82, 2362, 47, 1576, 56],
 [562, 10, 17, 82, 2362, 47, 1576, 56, 10],
 [562, 10, 17, 82, 2362, 47, 1576, 56, 10, 1577],
 [562, 10, 17, 82, 2362, 47, 1576, 56, 10, 1577, 3]]

In [17]:
padded_input_sequences = pad_sequences(input_sequences, maxlen = 50, padding='pre')

In [18]:
padded_input_sequences[0:5]

array([[   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,  562,   10],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,  562,   10,   17],
       [   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,  5

In [19]:
X = padded_input_sequences[:,:-1]
y = padded_input_sequences[:,-1]

In [20]:
X.shape

(38061, 49)

In [21]:
y.shape

(38061,)

In [22]:
from tensorflow.keras.utils import to_categorical
vocab_len = vocab_len + 1
y = to_categorical(y,num_classes=vocab_len)

In [23]:
y.shape

(38061, 4849)

In [24]:
X.shape

(38061, 49)

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [26]:
model = Sequential()
model.add(Embedding(vocab_len, 50, input_length = X.shape[0]))
model.add(LSTM(150))
model.add(Dense(vocab_len, activation='softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [27]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

In [30]:
history = model.fit(X, y, epochs= 100, batch_size = 128)

Epoch 1/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1289 - loss: 5.1936
Epoch 2/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.1345 - loss: 5.1013
Epoch 3/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.1348 - loss: 5.0224
Epoch 4/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1387 - loss: 4.9194
Epoch 5/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.1487 - loss: 4.7980
Epoch 6/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.1520 - loss: 4.7162
Epoch 7/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.1552 - loss: 4.6222
Epoch 8/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.1585 - loss: 4.5470
Epoch 9/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.1629 - loss: 4.4532
Epoch 10/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.1682 - loss: 4.3645
Epoch 11/100
298/298 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.1751 - loss: 4.2882
Epoch 12/100
298/298 ━━━━━━━━━━━━━━━━━

In [35]:
# jenerate output
import time
import numpy as np
text = "robort"

for i in range(10):
  # tokenize
  token_text = tokenizer.texts_to_sequences([text])[0]
  # padding
  padded_token_text = pad_sequences([token_text], maxlen=56, padding='pre')
  # predict
  pos = np.argmax(model.predict(padded_token_text))

  for word,index in tokenizer.word_index.items():
    if index == pos:
      text = text + " " + word
      print(text)
      time.sleep(2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
robort walked
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
robort walked oh
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
robort walked oh not
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
robort walked oh not brought
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
robort walked oh not brought them
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
robort walked oh not brought them to
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
robort walked oh not brought them to the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
robort walked oh not brought them to the n
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
robort walked oh not brought them to the n page
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
robort walked oh not brought them to the n page the


In [39]:
import pickle

# Save the model to a file using pickle.dump
with open('model.pkl', 'wb') as file:
    pickle.dump(model, file)

In [40]:
with open('tokenizer.pkl', 'wb') as file:
    pickle.dump(tokenizer, file)